# LeetCode 3924 — Minimum Threshold Path With Limited Heavy Edges bằng Python

- Link: https://leetcode.com/problems/minimum-threshold-path-with-limited-heavy-edges/
- Độ khó: **Hard**
- Chủ đề: Đồ thị vô hướng có trọng số, tìm kiếm nhị phân trên đáp án, BFS 0-1, Dijkstra trên đồ thị phân tầng

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumThreshold(self, n: int, edges: List[List[int]], source: int, target: int, k: int) -> int:
```

## 1. Mục tiêu học

Bài này là một ví dụ rất "chuẩn sách giáo khoa" của kỹ thuật **tìm kiếm nhị phân trên đáp án** kết hợp với **đường đi ngắn nhất có trọng số 0/1**. Sau notebook này bạn sẽ:

1. Biết cách biến câu hỏi "giá trị nhỏ nhất để tồn tại …" thành một **hàm kiểm tra đúng/sai đơn điệu** `ok(T)`.
2. Hiểu và cài đặt được **BFS 0-1** (dùng `deque`, cạnh trọng số 0 đẩy vào đầu, cạnh trọng số 1 đẩy vào cuối) để đếm số cạnh "nặng" ít nhất trên đường đi.
3. Hiểu vì sao chỉ cần thử các ngưỡng là **0 hoặc trọng số của một cạnh** (tập ứng viên hữu hạn).
4. Thấy một cách nhìn khác: đáp án = min trên mọi đường đi của **cạnh lớn thứ `k+1`** trên đường đó, và giải bằng **Dijkstra trên đồ thị phân tầng** `(đỉnh, số cạnh nặng đã dùng)`.
5. Tự viết bộ test đối chiếu giữa lời giải vét cạn đường đi đơn và các lời giải nhanh.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho đồ thị **vô hướng** `n` đỉnh (đánh số `0..n-1`), danh sách cạnh `edges[i] = [u, v, w]` (cạnh nối `u`–`v`, trọng số `w`). Cho thêm `source`, `target`, `k`.

Với một số nguyên `threshold` (ngưỡng):

- cạnh **nhẹ** nếu `w ≤ threshold`;
- cạnh **nặng** nếu `w > threshold`.

Một đường đi từ `source` đến `target` là **hợp lệ** nếu nó dùng **không quá `k` cạnh nặng**. Hãy tìm `threshold` **nhỏ nhất** (số nguyên, không âm vì trọng số ≥ 1 nên ngưỡng 0 đã là "mọi cạnh đều nặng") sao cho tồn tại ít nhất một đường đi hợp lệ. Nếu không có đường đi nào từ `source` tới `target` thì trả về `-1`.

**Ví dụ 1:** `n = 6, edges = [[0,1,5],[1,2,3],[3,4,4],[4,5,1],[1,4,2]], source = 0, target = 3, k = 1` → `4`.
Với ngưỡng 4, chỉ cạnh `[0,1,5]` là nặng; đường `0 → 1 → 4 → 3` dùng đúng 1 cạnh nặng.

**Ví dụ 2:** `n = 6, edges = [[0,1,3],[1,2,4],[3,4,5],[4,5,6]], source = 0, target = 4, k = 1` → `-1` (0 và 4 không liên thông).

**Ví dụ 3:** `n = 4, edges = [[0,1,2],[1,2,2],[2,3,2],[3,0,2]], source = 0, target = 0, k = 0` → `0` (đứng yên, không đi cạnh nào).

**Ràng buộc:**

- `1 ≤ n ≤ 10^3`, `0 ≤ edges.length ≤ 10^3`
- `0 ≤ u, v ≤ n-1`, `1 ≤ w ≤ 10^9`
- `0 ≤ source, target ≤ n-1`, `0 ≤ k ≤ edges.length`

In [1]:
from typing import List
from collections import deque
import heapq, random

examples = [
    (6, [[0,1,5],[1,2,3],[3,4,4],[4,5,1],[1,4,2]], 0, 3, 1, 4),
    (6, [[0,1,3],[1,2,4],[3,4,5],[4,5,6]], 0, 4, 1, -1),
    (4, [[0,1,2],[1,2,2],[2,3,2],[3,0,2]], 0, 0, 0, 0),
]
for n, edges, s, t, k, expected in examples:
    print(f"n={n}, source={s}, target={t}, k={k} -> kỳ vọng {expected}")

n=6, source=0, target=3, k=1 -> kỳ vọng 4
n=6, source=0, target=4, k=1 -> kỳ vọng -1
n=4, source=0, target=0, k=0 -> kỳ vọng 0


## 3. Kiến thức nền

Ta cần 4 viên gạch: (1) danh sách kề cho đồ thị vô hướng, (2) `deque` và BFS 0-1, (3) tìm kiếm nhị phân trên mảng ứng viên đã sắp xếp, (4) `heapq` cho Dijkstra.

### 3.1. Danh sách kề (adjacency list) cho đồ thị vô hướng

Mỗi cạnh `[u, v, w]` vô hướng nên ta thêm **hai chiều**: `u → v` và `v → u`. `adj[u]` là danh sách các cặp `(láng giềng, trọng số)`.

In [2]:
def build_adj(n, edges):
    adj = [[] for _ in range(n)]
    for u, v, w in edges:
        adj[u].append((v, w))
        adj[v].append((u, w))
    return adj

adj = build_adj(6, examples[0][1])
for u in range(6):
    print(u, adj[u])

0 [(1, 5)]
1 [(0, 5), (2, 3), (4, 2)]
2 [(1, 3)]
3 [(4, 4)]
4 [(3, 4), (5, 1), (1, 2)]
5 [(4, 1)]


### 3.2. `deque` và BFS 0-1

`collections.deque` cho phép thêm/bớt ở **cả hai đầu** trong `O(1)`: `appendleft`, `append`, `popleft`.

**BFS 0-1** tìm đường đi ngắn nhất khi mọi trọng số chỉ là `0` hoặc `1`:

- lấy đỉnh `u` ở đầu deque;
- với cạnh trọng số `0` tới `v` mà cải thiện được `dist[v]` → `appendleft(v)` (vì `dist[v] = dist[u]`, nó "ngang hàng" với `u`);
- với cạnh trọng số `1` → `append(v)` (xa hơn một bậc).

Nhờ vậy deque luôn chứa các đỉnh có khoảng cách `d` ở đầu và `d+1` ở cuối — giống Dijkstra nhưng không cần heap, độ phức tạp `O(V + E)`.

In [3]:
d = deque([2])
d.appendleft(1)   # thêm vào đầu
d.append(3)       # thêm vào cuối
print(list(d), d.popleft(), list(d))

def zero_one_bfs(n, adj01, src):
    """adj01[u] = [(v, c)] với c ∈ {0, 1}. Trả về mảng khoảng cách."""
    INF = float('inf')
    dist = [INF] * n
    dist[src] = 0
    dq = deque([src])
    while dq:
        u = dq.popleft()
        for v, c in adj01[u]:
            nd = dist[u] + c
            if nd < dist[v]:
                dist[v] = nd
                if c == 0:
                    dq.appendleft(v)
                else:
                    dq.append(v)
    return dist

adj01 = [[(1, 1), (2, 0)], [(0, 1), (3, 0)], [(0, 0), (3, 1)], [(1, 0), (2, 1)]]
print(zero_one_bfs(4, adj01, 0))

[1, 2, 3] 1 [2, 3]
[0, 1, 0, 1]


### 3.3. Tìm kiếm nhị phân "giá trị nhỏ nhất thỏa điều kiện"

Nếu hàm `ok(x)` **đơn điệu** — một khi đúng thì đúng mãi khi `x` tăng — thì tập `{x : ok(x)}` có dạng `F F F … F T T T …`. Ta tìm chữ `T` đầu tiên bằng nhị phân. Mẫu dưới đây chạy trên **mảng ứng viên đã sắp xếp** `cand`.

In [4]:
def first_true(cand, ok):
    lo, hi = 0, len(cand) - 1        # giả sử ok(cand[-1]) đúng
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(cand[mid]):
            hi = mid                 # mid đúng -> đáp án ở mid hoặc bên trái
        else:
            lo = mid + 1             # mid sai -> đáp án ở bên phải
    return cand[lo]

cand = [0, 3, 7, 10, 15, 22]
print(first_true(cand, lambda x: x >= 9))   # 10
print(first_true(cand, lambda x: x >= 0))   # 0

10
0


### 3.4. `heapq` — hàng đợi ưu tiên

`heapq.heappush(h, item)` và `heapq.heappop(h)` luôn lấy ra phần tử **nhỏ nhất**. Tuple được so sánh theo thành phần đầu tiên trước, nên ta đẩy `(khoảng_cách, đỉnh, ...)`.

In [5]:
h = []
for item in [(5, 'a'), (1, 'b'), (3, 'c')]:
    heapq.heappush(h, item)
print([heapq.heappop(h) for _ in range(3)])

[(1, 'b'), (3, 'c'), (5, 'a')]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy tính tay ví dụ 1 (`source = 0`, `target = 3`, `k = 1`). Các cạnh: `0-1 (5)`, `1-2 (3)`, `3-4 (4)`, `4-5 (1)`, `1-4 (2)`.

Chỉ có **một** đường đi đơn từ 0 tới 3: `0 -(5)- 1 -(2)- 4 -(4)- 3`. Trọng số trên đường: `[5, 2, 4]`.

- Ta được phép để **tối đa 1 cạnh nặng**. Muốn ngưỡng nhỏ nhất, cạnh nặng đó nên là **cạnh lớn nhất** (5).
- Hai cạnh còn lại `[2, 4]` phải nhẹ → `threshold ≥ 4`.
- Vậy ngưỡng tối thiểu cho đường này là `4` = **cạnh lớn thứ 2** trên đường.

Tổng quát: với một đường đi `P`, nếu sắp trọng số giảm dần `a1 ≥ a2 ≥ …`, thì ngưỡng nhỏ nhất làm `P` hợp lệ là `a_{k+1}` (cạnh lớn thứ `k+1`), hoặc `0` nếu `P` có `≤ k` cạnh. Đáp án của bài = **min** giá trị này trên mọi đường đi.

Một quan sát nữa: khi `threshold` **tăng**, cạnh chỉ có thể đổi từ nặng → nhẹ, không bao giờ ngược lại. Nên nếu ngưỡng `T` đã có đường hợp lệ thì mọi `T' > T` cũng có. Đây chính là tính **đơn điệu** để tìm nhị phân.

In [6]:
def threshold_for_path(weights, k):
    """Ngưỡng nhỏ nhất để một đường cố định với các trọng số `weights` dùng ≤ k cạnh nặng."""
    if len(weights) <= k:
        return 0
    return sorted(weights, reverse=True)[k]

print(threshold_for_path([5, 2, 4], 1))   # 4
print(threshold_for_path([5, 2, 4], 0))   # 5
print(threshold_for_path([5, 2, 4], 3))   # 0

4
5
0


## 5. Cách 1 — Brute Force: vét cạn mọi đường đi đơn

**Ý tưởng.** Dùng DFS liệt kê **mọi đường đi đơn** (không lặp đỉnh) từ `source` tới `target`. Với mỗi đường, tính `threshold_for_path`. Lấy min. Nếu không có đường nào → `-1`.

Vì sao chỉ cần đường đi **đơn**? Nếu một lộ trình đi vòng (lặp đỉnh), ta cắt bỏ vòng lặp thì số cạnh nặng chỉ **giảm hoặc giữ nguyên** với bất kỳ ngưỡng nào, nên đường đơn không bao giờ tệ hơn.

Trường hợp `source == target`: đường rỗng, 0 cạnh → đáp án `0`.

In [7]:
def brute_paths(n, edges, source, target, k):
    adj = build_adj(n, edges)
    best = [None]
    visited = [False] * n

    def dfs(u, weights):
        if u == target:
            val = threshold_for_path(weights, k)
            if best[0] is None or val < best[0]:
                best[0] = val
            return
        for v, w in adj[u]:
            if not visited[v]:
                visited[v] = True
                weights.append(w)
                dfs(v, weights)
                weights.pop()
                visited[v] = False

    visited[source] = True
    dfs(source, [])
    return -1 if best[0] is None else best[0]

for n, e, s, t, k, exp in examples:
    print(brute_paths(n, e, s, t, k), "kỳ vọng", exp)

4 kỳ vọng 4
-1 kỳ vọng -1
0 kỳ vọng 0


**Độ phức tạp.** Số đường đi đơn có thể là **hàm mũ** theo `n` (ví dụ đồ thị đầy đủ có cỡ `(n-2)!` đường). Với `n = 1000` hoàn toàn không khả thi.

**Nút thắt:** ta đang xét **từng đường đi**. Cần một cách "gộp" mọi đường lại — tức là trả lời câu hỏi *"với ngưỡng T có đường hợp lệ không?"* bằng một thuật toán đường đi ngắn nhất.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — Đổi câu hỏi tối ưu thành câu hỏi quyết định.** Cố định `T`. Gán **chi phí 0** cho cạnh nhẹ (`w ≤ T`) và **chi phí 1** cho cạnh nặng (`w > T`). Khi đó:

> Tồn tại đường hợp lệ với ngưỡng `T` **⇔** đường đi ngắn nhất (theo chi phí 0/1) từ `source` tới `target` có độ dài `≤ k`.

Đường ngắn nhất 0/1 tính bằng **BFS 0-1** trong `O(n + m)`.

**Insight 2 — Đơn điệu.** Nếu `ok(T)` đúng thì `ok(T')` đúng với mọi `T' ≥ T` (cạnh nặng chỉ có thể thành nhẹ). Vậy tìm nhị phân được.

**Insight 3 — Tập ứng viên hữu hạn.** `ok(T)` chỉ thay đổi khi `T` vượt qua một trọng số cạnh. Do đó đáp án luôn thuộc tập `{0} ∪ {w_i}`. Ta sắp xếp tập này và tìm nhị phân **trên chỉ số**.

**Quy tắc "nếu … thì …":**

- Nếu `source == target` → trả `0`.
- Nếu `target` không tới được từ `source` (kể cả khi mọi cạnh đều nhẹ) → trả `-1`.
- Ngược lại, `ok(max_w)` chắc chắn đúng (mọi cạnh nhẹ, 0 cạnh nặng ≤ k) → tìm nhị phân trên ứng viên.

## 7. Các cách tiếp theo

Ta trình bày thêm 3 cách:

- **Cách 2:** thử **mọi** ngưỡng ứng viên theo thứ tự tăng dần, mỗi lần chạy BFS 0-1 (`O(m · (n+m))`).
- **Cách 3:** **tìm nhị phân** trên ứng viên + BFS 0-1 (`O((n+m) log m)`) — đây là lời giải nộp chính.
- **Cách 4:** **Dijkstra trên đồ thị phân tầng** `(đỉnh, j = số cạnh nặng đã dùng)`, tối thiểu hóa *cạnh nhẹ lớn nhất* — cách nhìn "minimax có k lần bỏ qua".

### 7.1. Cách 2 — Thử tuần tự mọi ngưỡng ứng viên

**Giải thích từng đoạn:**

1. `heavy_dist(T)`: dựng chi phí 0/1 theo `T` "trên đường đi" (không cần dựng lại danh sách kề, chỉ so `w > T` khi duyệt), chạy BFS 0-1, trả về `dist[target]`.
2. Kiểm tra liên thông bằng `heavy_dist(max_w)` (mọi cạnh nhẹ): nếu `∞` → `-1`.
3. Duyệt ứng viên tăng dần; ngưỡng đầu tiên cho `dist ≤ k` là đáp án.

In [8]:
def heavy_dist(n, adj, source, target, T):
    INF = float('inf')
    dist = [INF] * n
    dist[source] = 0
    dq = deque([source])
    while dq:
        u = dq.popleft()
        du = dist[u]
        for v, w in adj[u]:
            c = 1 if w > T else 0
            if du + c < dist[v]:
                dist[v] = du + c
                if c:
                    dq.append(v)
                else:
                    dq.appendleft(v)
    return dist[target]

def linear_scan(n, edges, source, target, k):
    if source == target:
        return 0
    adj = build_adj(n, edges)
    cand = sorted({0} | {w for _, _, w in edges})
    if heavy_dist(n, adj, source, target, cand[-1]) == float('inf'):
        return -1
    for T in cand:
        if heavy_dist(n, adj, source, target, T) <= k:
            return T

for n, e, s, t, k, exp in examples:
    print(linear_scan(n, e, s, t, k), "kỳ vọng", exp)

4 kỳ vọng 4
-1 kỳ vọng -1
0 kỳ vọng 0


Code nộp LeetCode cho Cách 2 (đủ nhanh vì `m ≤ 1000`: khoảng `1000 × 3000` thao tác ≈ `3·10^6`):

In [9]:
class SolutionLinear:
    def minimumThreshold(self, n: int, edges: List[List[int]], source: int, target: int, k: int) -> int:
        if source == target:
            return 0
        adj = [[] for _ in range(n)]
        for u, v, w in edges:
            adj[u].append((v, w))
            adj[v].append((u, w))

        def dist_to_target(T):
            INF = float('inf')
            dist = [INF] * n
            dist[source] = 0
            dq = deque([source])
            while dq:
                u = dq.popleft()
                for v, w in adj[u]:
                    c = 1 if w > T else 0
                    if dist[u] + c < dist[v]:
                        dist[v] = dist[u] + c
                        (dq.append if c else dq.appendleft)(v)
            return dist[target]

        cand = sorted({0} | {w for _, _, w in edges})
        if dist_to_target(cand[-1]) == float('inf'):
            return -1
        for T in cand:
            if dist_to_target(T) <= k:
                return T
        return -1

print([SolutionLinear().minimumThreshold(*ex[:5]) for ex in examples])

[4, -1, 0]


**Độ phức tạp Cách 2:** `O(m · (n + m))` thời gian, `O(n + m)` bộ nhớ. Ổn với ràng buộc hiện tại, nhưng nếu `m` lên `10^5` thì quá chậm.

### 7.2. Cách 3 — Tìm nhị phân trên ứng viên + BFS 0-1 (lời giải chính)

**Giải thích từng đoạn:**

1. Xử lý `source == target` → `0`.
2. `cand = sorted({0} ∪ {w})`. Nếu `ok(cand[-1])` sai thì không liên thông → `-1`.
3. Tìm nhị phân `lo, hi` trên chỉ số của `cand`: nếu `ok(cand[mid])` → `hi = mid`, ngược lại `lo = mid + 1`.
4. Trả về `cand[lo]`.

Hàm `ok(T)` có thể **dừng sớm**: BFS 0-1 lấy đỉnh theo thứ tự khoảng cách tăng dần, nên khi lấy ra `target` thì `dist[target]` đã tối ưu.

In [10]:
def binary_search_sol(n, edges, source, target, k):
    if source == target:
        return 0
    adj = build_adj(n, edges)
    cand = sorted({0} | {w for _, _, w in edges})

    def ok(T):
        return heavy_dist(n, adj, source, target, T) <= k

    if not ok(cand[-1]):
        return -1
    lo, hi = 0, len(cand) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(cand[mid]):
            hi = mid
        else:
            lo = mid + 1
    return cand[lo]

for n, e, s, t, k, exp in examples:
    print(binary_search_sol(n, e, s, t, k), "kỳ vọng", exp)

4 kỳ vọng 4
-1 kỳ vọng -1
0 kỳ vọng 0


**Độ phức tạp Cách 3:** mỗi lần `ok` là `O(n + m)`, gọi `O(log m)` lần, cộng sắp xếp `O(m log m)` → tổng `O((n + m) log m)`. Bộ nhớ `O(n + m)`.

### 7.3. Cách 4 — Dijkstra trên đồ thị phân tầng (minimax với tối đa k lần "bỏ qua")

**Cách nhìn khác.** Theo mục 4, đáp án = `min` trên mọi đường của "cạnh lớn thứ `k+1`". Tương đương: trên đường đi, ta được **chọn tối đa `k` cạnh để "miễn phí"** (gọi là nặng), các cạnh còn lại tính bằng **max**. Muốn tối thiểu hóa cái max đó.

**Trạng thái:** `best[v][j]` = giá trị max nhỏ nhất của các cạnh **không** được miễn, trên đường từ `source` tới `v` đã dùng `j` lần miễn.

**Chuyển trạng thái** qua cạnh `(u, v, w)`:

- không miễn: `best[v][j] ← max(best[u][j], w)`;
- miễn (nếu `j < k`): `best[v][j+1] ← best[u][j]`.

Vì "giá" là `max` (không giảm khi đi tiếp), Dijkstra vẫn đúng (bài toán *bottleneck path*). Đáp án = `min_j best[target][j]`.

Số trạng thái `n · (k+1)` có thể lên `10^6`, nên cách này chậm hơn Cách 3 trong Python nhưng rất hữu ích để hiểu bài và để đối chiếu. Mẹo nhỏ: ta có thể giới hạn `k` bởi `n - 1` vì đường đơn có tối đa `n - 1` cạnh.

In [11]:
def layered_dijkstra(n, edges, source, target, k):
    if source == target:
        return 0
    adj = build_adj(n, edges)
    k = min(k, n - 1)
    INF = float('inf')
    best = [[INF] * (k + 1) for _ in range(n)]
    best[source][0] = 0
    h = [(0, source, 0)]
    while h:
        cur, u, j = heapq.heappop(h)
        if cur > best[u][j]:
            continue
        if u == target:
            return cur               # lần đầu lấy ra target là tối ưu
        for v, w in adj[u]:
            nv = max(cur, w)          # không miễn
            if nv < best[v][j]:
                best[v][j] = nv
                heapq.heappush(h, (nv, v, j))
            if j < k and cur < best[v][j + 1]:   # miễn cạnh này
                best[v][j + 1] = cur
                heapq.heappush(h, (cur, v, j + 1))
    return -1

for n, e, s, t, k, exp in examples:
    print(layered_dijkstra(n, e, s, t, k), "kỳ vọng", exp)

4 kỳ vọng 4
-1 kỳ vọng -1
0 kỳ vọng 0


Code nộp LeetCode cho Cách 4 (có thể chậm ở ca xấu nhất `n = m = k = 1000`, dùng để học/đối chiếu):

In [12]:
class SolutionLayered:
    def minimumThreshold(self, n: int, edges: List[List[int]], source: int, target: int, k: int) -> int:
        if source == target:
            return 0
        adj = [[] for _ in range(n)]
        for u, v, w in edges:
            adj[u].append((v, w))
            adj[v].append((u, w))
        k = min(k, n - 1)
        INF = float('inf')
        best = [[INF] * (k + 1) for _ in range(n)]
        best[source][0] = 0
        h = [(0, source, 0)]
        while h:
            cur, u, j = heapq.heappop(h)
            if cur > best[u][j]:
                continue
            if u == target:
                return cur
            for v, w in adj[u]:
                nv = cur if cur > w else w
                if nv < best[v][j]:
                    best[v][j] = nv
                    heapq.heappush(h, (nv, v, j))
                if j < k and cur < best[v][j + 1]:
                    best[v][j + 1] = cur
                    heapq.heappush(h, (cur, v, j + 1))
        return -1

print([SolutionLayered().minimumThreshold(*ex[:5]) for ex in examples])

[4, -1, 0]


**Độ phức tạp Cách 4:** `O(k · m · log(k · m))` thời gian, `O(n · k)` bộ nhớ.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1. Ứng viên `cand = [0, 1, 2, 3, 4, 5]`.

| bước | lo | hi | mid | T = cand[mid] | dist[target] (số cạnh nặng) | ok? |
|---|---|---|---|---|---|---|
| kiểm tra liên thông | – | – | – | 5 | 0 | ✔ |
| 1 | 0 | 5 | 2 | 2 | 2 (cạnh 5 và 4 trên đường 0-1-4-3 đều nặng) | ✘ |
| 2 | 3 | 5 | 4 | 4 | 1 | ✔ |
| 3 | 3 | 4 | 3 | 3 | 2 | ✘ |
| kết thúc | 4 | 4 | – | **4** | | |

(Bảng trên tóm tắt; hàm trace dưới đây in đúng giá trị thực tế.)

In [13]:
def trace_binary(n, edges, source, target, k):
    adj = build_adj(n, edges)
    cand = sorted({0} | {w for _, _, w in edges})
    print("cand =", cand)
    print("liên thông? dist với T = max:", heavy_dist(n, adj, source, target, cand[-1]))
    lo, hi = 0, len(cand) - 1
    step = 0
    while lo < hi:
        mid = (lo + hi) // 2
        d = heavy_dist(n, adj, source, target, cand[mid])
        good = d <= k
        step += 1
        print(f"bước {step}: lo={lo} hi={hi} mid={mid} T={cand[mid]} heavy={d} ok={good}")
        if good:
            hi = mid
        else:
            lo = mid + 1
    print("đáp án:", cand[lo])

trace_binary(*examples[0][:5])

cand = [0, 1, 2, 3, 4, 5]
liên thông? dist với T = max: 0
bước 1: lo=0 hi=5 mid=2 T=2 heavy=2 ok=False
bước 2: lo=3 hi=5 mid=4 T=4 heavy=1 ok=True
bước 3: lo=3 hi=4 mid=3 T=3 heavy=2 ok=False
đáp án: 4


Trace bên trong **một lần BFS 0-1** với `T = 4` (cạnh nặng duy nhất là `0-1`): theo dõi deque và mảng `dist`.

In [14]:
def trace_01bfs(n, edges, source, T):
    adj = build_adj(n, edges)
    INF = float('inf')
    dist = [INF] * n
    dist[source] = 0
    dq = deque([source])
    while dq:
        u = dq.popleft()
        print(f"lấy u={u} (dist={dist[u]}), deque còn {list(dq)}")
        for v, w in adj[u]:
            c = 1 if w > T else 0
            if dist[u] + c < dist[v]:
                dist[v] = dist[u] + c
                (dq.append if c else dq.appendleft)(v)
                print(f"   cạnh {u}-{v} w={w} -> chi phí {c}; dist[{v}]={dist[v]}; deque={list(dq)}")
    print("dist cuối:", dist)

trace_01bfs(6, examples[0][1], 0, 4)

lấy u=0 (dist=0), deque còn []
   cạnh 0-1 w=5 -> chi phí 1; dist[1]=1; deque=[1]
lấy u=1 (dist=1), deque còn []
   cạnh 1-2 w=3 -> chi phí 0; dist[2]=1; deque=[2]
   cạnh 1-4 w=2 -> chi phí 0; dist[4]=1; deque=[4, 2]
lấy u=4 (dist=1), deque còn [2]
   cạnh 4-3 w=4 -> chi phí 0; dist[3]=1; deque=[3, 2]
   cạnh 4-5 w=1 -> chi phí 0; dist[5]=1; deque=[5, 3, 2]
lấy u=5 (dist=1), deque còn [3, 2]
lấy u=3 (dist=1), deque còn [2]
lấy u=2 (dist=1), deque còn []
dist cuối: [0, 1, 1, 1, 1, 1]


## 9. Code nộp LeetCode

Lời giải chính (Cách 3): tìm nhị phân trên tập ứng viên + BFS 0-1 có dừng sớm.

In [15]:
class Solution:
    def minimumThreshold(self, n: int, edges: List[List[int]], source: int, target: int, k: int) -> int:
        if source == target:
            return 0
        adj = [[] for _ in range(n)]
        for u, v, w in edges:
            adj[u].append((v, w))
            adj[v].append((u, w))
        INF = float('inf')

        def ok(T):
            dist = [INF] * n
            dist[source] = 0
            dq = deque([source])
            while dq:
                u = dq.popleft()
                du = dist[u]
                if u == target:
                    return du <= k
                if du > k:
                    return False          # mọi đỉnh còn lại trong deque đều có dist ≥ du > k
                for v, w in adj[u]:
                    if w > T:
                        if du + 1 < dist[v]:
                            dist[v] = du + 1
                            dq.append(v)
                    elif du < dist[v]:
                        dist[v] = du
                        dq.appendleft(v)
            return False

        cand = sorted({0} | {w for _, _, w in edges})
        if not ok(cand[-1]):
            return -1
        lo, hi = 0, len(cand) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if ok(cand[mid]):
                hi = mid
            else:
                lo = mid + 1
        return cand[lo]

sol = Solution()
for n, e, s, t, k, exp in examples:
    got = sol.minimumThreshold(n, e, s, t, k)
    print(got, "kỳ vọng", exp, "✔" if got == exp else "✘")

4 kỳ vọng 4 ✔
-1 kỳ vọng -1 ✔
0 kỳ vọng 0 ✔


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (đổi bài toán).** Với `T` cố định, gán chi phí `c(e) = [w_e > T]`. Chi phí một đường = số cạnh nặng trên đường. Tồn tại đường hợp lệ ⇔ `min_P cost(P) ≤ k` ⇔ khoảng cách 0/1 ngắn nhất `≤ k`. BFS 0-1 tính đúng khoảng cách vì nó giữ bất biến: deque luôn có dạng `[các đỉnh khoảng cách d …, các đỉnh khoảng cách d+1 …]` (giống Dijkstra với chỉ hai mức).

**Bổ đề 2 (đơn điệu).** Nếu `T ≤ T'` thì với mọi cạnh, `[w > T'] ≤ [w > T]`. Nên cost theo `T'` ≤ cost theo `T` trên mọi đường. Do đó `ok(T) ⇒ ok(T')`.

**Bổ đề 3 (ứng viên).** Giả sử đáp án tối ưu là `T*`. Nếu `T*` không thuộc `{0} ∪ {w_i}`, đặt `T0` = phần tử lớn nhất của tập đó mà `< T*` (luôn tồn tại vì có 0 và `T* > 0`). Không có trọng số nào trong `(T0, T*]`, nên mọi cạnh có cùng phân loại với `T0` và `T*` ⇒ `ok(T0)` đúng, mâu thuẫn tính nhỏ nhất. Vậy `T*` là một ứng viên.

**Ghép lại.** Nếu `target` không tới được thì không ngưỡng nào thỏa → `-1`. Ngược lại `ok(max ứng viên)` đúng (mọi cạnh nhẹ). Tìm nhị phân trên dãy `F…FT…T` trả về `T` đầu tiên — chính là đáp án. Trường hợp `source = target` cho đường 0 cạnh nên đáp án `0` (ứng viên nhỏ nhất).

## 11. Phân tích Time / Space Complexity

Đặt `n` = số đỉnh, `m` = số cạnh.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn đường đơn | hàm mũ (tệ nhất cỡ `(n-2)!`) | `O(n + m)` |
| 2. Thử mọi ngưỡng | `O(m · (n + m))` | `O(n + m)` |
| 3. Nhị phân + BFS 0-1 | `O(m log m + (n + m) log m)` | `O(n + m)` |
| 4. Dijkstra phân tầng | `O(k · m · log(k m))` với `k ≤ n-1` | `O(n · k)` |

Với `n, m ≤ 1000`: Cách 3 chỉ khoảng `11` lần BFS, mỗi lần ~`3000` thao tác — cực nhanh.

## 12. Bộ test

Gồm: ví dụ của đề, ca biên tự dựng, và đối chiếu ngẫu nhiên (seed cố định) giữa 4 cách. Ca biên tự dựng đã được kiểm chứng bằng vét cạn.

In [16]:
def run_all(n, e, s, t, k):
    return (brute_paths(n, e, s, t, k), linear_scan(n, e, s, t, k),
            binary_search_sol(n, e, s, t, k), layered_dijkstra(n, e, s, t, k),
            Solution().minimumThreshold(n, e, s, t, k))

# 12.1 Ví dụ đề
for n, e, s, t, k, exp in examples:
    r = run_all(n, e, s, t, k)
    assert len(set(r)) == 1 and r[0] == exp, (r, exp)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [17]:
# 12.2 Ca biên (tự dựng, đã kiểm chứng bằng vét cạn)
edge_cases = [
    (1, [], 0, 0, 0, 0),                          # một đỉnh, không cạnh
    (2, [], 0, 1, 0, -1),                         # hai đỉnh rời nhau
    (2, [[0, 1, 7]], 0, 1, 0, 7),                 # k = 0: phải làm nhẹ mọi cạnh
    (2, [[0, 1, 7]], 0, 1, 1, 0),                 # k ≥ số cạnh trên đường -> 0
    (3, [[0, 1, 5], [1, 2, 5], [0, 2, 9]], 0, 2, 0, 5),   # đường dài nhưng cạnh nhỏ thắng
    (3, [[0, 1, 5], [1, 2, 5], [0, 2, 9]], 0, 2, 1, 0),   # cạnh trực tiếp miễn luôn
    (3, [[0, 1, 4], [0, 1, 2], [1, 2, 3]], 0, 2, 0, 3),   # cạnh song song
    (3, [[0, 0, 1], [0, 1, 6], [1, 2, 8]], 0, 2, 1, 6),   # có self-loop
]
for n, e, s, t, k, exp in edge_cases:
    r = run_all(n, e, s, t, k)
    assert len(set(r)) == 1 and r[0] == exp, (n, e, s, t, k, r, exp)
print("Ca biên: OK")

Ca biên: OK


In [18]:
# 12.3 Phản ví dụ cho ý tưởng tham lam "luôn đi đường ít cạnh nhất"
# Đường ít cạnh nhất 0-3 trực tiếp có w = 100; đường dài 0-1-2-3 có các cạnh nhỏ.
n, e = 4, [[0, 3, 100], [0, 1, 1], [1, 2, 2], [2, 3, 3]]
print("k=0:", Solution().minimumThreshold(n, e, 0, 3, 0), "(đường dài cho 3, đường ngắn cho 100)")
assert Solution().minimumThreshold(n, e, 0, 3, 0) == 3
assert Solution().minimumThreshold(n, e, 0, 3, 1) == 0

k=0: 3 (đường dài cho 3, đường ngắn cho 100)


In [19]:
# 12.4 Đối chiếu ngẫu nhiên giữa các cách (đồ thị nhỏ để vét cạn được)
rng = random.Random(3924)
for it in range(400):
    n = rng.randint(1, 7)
    m = rng.randint(0, 9)
    e = [[rng.randrange(n), rng.randrange(n), rng.randint(1, 12)] for _ in range(m)]
    s, t = rng.randrange(n), rng.randrange(n)
    k = rng.randint(0, m)
    r = run_all(n, e, s, t, k)
    assert len(set(r)) == 1, (n, e, s, t, k, r)
print("400 test ngẫu nhiên: OK")

400 test ngẫu nhiên: OK


In [20]:
# 12.5 Test lớn đúng ràng buộc (n = m = 1000), so Cách 2 và Cách 3, kiểm tra thời gian
import time
rng = random.Random(1)
n = 1000
e = [[rng.randrange(n), rng.randrange(n), rng.randint(1, 10**9)] for _ in range(1000)]
k = 3
t0 = time.time(); a = Solution().minimumThreshold(n, e, 0, 999, k); t1 = time.time()
b = linear_scan(n, e, 0, 999, k)
print("đáp án:", a, "| khớp Cách 2:", a == b, "| thời gian Cách 3 < 1s:", (t1 - t0) < 1)

đáp án: -1 | khớp Cách 2: True | thời gian Cách 3 < 1s: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trường hợp `source == target`

Nếu chỉ tìm nhị phân trên `{w_i}` mà không có `0`, ví dụ 3 sẽ trả `2` thay vì `0`. Còn nếu `edges` rỗng thì `cand` rỗng → lỗi chỉ số.

In [21]:
def wrong_no_zero(n, edges, source, target, k):
    adj = build_adj(n, edges)
    cand = sorted({w for _, _, w in edges})          # SAI: thiếu 0, và có thể rỗng
    if not cand:
        return -1
    lo, hi = 0, len(cand) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if heavy_dist(n, adj, source, target, cand[mid]) <= k:
            hi = mid
        else:
            lo = mid + 1
    return cand[lo] if heavy_dist(n, adj, source, target, cand[lo]) <= k else -1

print("sai:", wrong_no_zero(*examples[2][:5]), "| đúng:", Solution().minimumThreshold(*examples[2][:5]))
print("sai (1 đỉnh, không cạnh):", wrong_no_zero(1, [], 0, 0, 0), "| đúng:", Solution().minimumThreshold(1, [], 0, 0, 0))

sai: 2 | đúng: 0
sai (1 đỉnh, không cạnh): -1 | đúng: 0


**Sửa:** luôn thêm `0` vào tập ứng viên và xử lý `source == target` trước.

### Lỗi 2 — Dùng BFS thường (đếm số cạnh) thay vì BFS 0-1

BFS thường coi mọi cạnh có giá 1, nên nó đếm **số cạnh**, không phải **số cạnh nặng**. Kết quả là `ok(T)` gần như không phụ thuộc `T`.

In [22]:
def wrong_plain_bfs_ok(n, edges, source, target, k, T):
    adj = build_adj(n, edges)
    INF = float('inf')
    dist = [INF] * n; dist[source] = 0
    dq = deque([source])
    while dq:
        u = dq.popleft()
        for v, w in adj[u]:
            if dist[v] == INF:                      # SAI: bỏ qua phân loại nhẹ/nặng
                dist[v] = dist[u] + (1 if w > T else 0)
                dq.append(v)
    return dist[target] <= k

n, e = 4, [[0, 3, 100], [0, 1, 1], [1, 2, 2], [2, 3, 3]]
print("BFS thường, T=3, k=0:", wrong_plain_bfs_ok(n, e, 0, 3, 0, 3), "(phải là True)")
adj = build_adj(n, e)
print("BFS 0-1, T=3, k=0:", heavy_dist(n, adj, 0, 3, 3) <= 0)

BFS thường, T=3, k=0: False (phải là True)
BFS 0-1, T=3, k=0: True


**Vì sao sai:** BFS thường "chốt" `dist[3]` ngay khi thấy cạnh trực tiếp `0-3` (nặng), không cho đường dài toàn cạnh nhẹ cập nhật lại. **Sửa:** dùng BFS 0-1 (hoặc Dijkstra) — cho phép cập nhật khi tìm được giá trị nhỏ hơn.

### Lỗi 3 — Sai điều kiện nhẹ/nặng (`<` thay vì `≤`)

Đề nói cạnh **nhẹ** khi `w ≤ threshold`. Nếu viết `w >= T` là nặng thì cạnh có `w == T` bị tính nặng → đáp án lệch lên một ứng viên.

In [23]:
def heavy_dist_wrong(n, adj, source, target, T):
    INF = float('inf')
    dist = [INF] * n; dist[source] = 0
    dq = deque([source])
    while dq:
        u = dq.popleft()
        for v, w in adj[u]:
            c = 1 if w >= T else 0                  # SAI: phải là w > T
            if dist[u] + c < dist[v]:
                dist[v] = dist[u] + c
                (dq.append if c else dq.appendleft)(v)
    return dist[target]

adj = build_adj(6, examples[0][1])
print("sai: với T=4 số cạnh nặng =", heavy_dist_wrong(6, adj, 0, 3, 4), "-> ok(4) =", heavy_dist_wrong(6, adj, 0, 3, 4) <= 1)
print("đúng: với T=4 số cạnh nặng =", heavy_dist(6, adj, 0, 3, 4))

sai: với T=4 số cạnh nặng = 2 -> ok(4) = False
đúng: với T=4 số cạnh nặng = 1


### Lỗi 4 — Tìm nhị phân trên khoảng `[0, 10^9]` nhưng viết sai vòng lặp

Tìm nhị phân trên giá trị cũng đúng (khoảng 30 lần BFS), nhưng dễ sai biên: `while lo <= hi` cùng `hi = mid` gây **lặp vô hạn**. Mẫu an toàn: `while lo < hi`, `hi = mid` khi đúng, `lo = mid + 1` khi sai.

In [24]:
def value_binary_search(n, edges, source, target, k):
    if source == target:
        return 0
    adj = build_adj(n, edges)
    if heavy_dist(n, adj, source, target, 10**9) == float('inf'):
        return -1
    lo, hi = 0, 10**9
    while lo < hi:                       # ĐÚNG: lo < hi
        mid = (lo + hi) // 2
        if heavy_dist(n, adj, source, target, mid) <= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

print([value_binary_search(*ex[:5]) for ex in examples])

[4, -1, 0]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Câu hỏi **"giá trị nhỏ nhất/lớn nhất sao cho tồn tại …"** + điều kiện đơn điệu ⇒ nghĩ ngay tới **tìm nhị phân trên đáp án**.
- **"Tối đa k phần tử được miễn/đặc biệt"** trên đường đi ⇒ hai hướng: (a) cố định ngưỡng rồi đếm số phần tử đặc biệt bằng **BFS 0-1**, (b) **đồ thị phân tầng** `(đỉnh, số lần dùng)`.
- Trọng số chỉ 0/1 ⇒ BFS 0-1 thay Dijkstra (`O(V+E)`, code ngắn).
- Ứng viên đáp án thường là **các giá trị có trong dữ liệu** (trọng số cạnh) — nhị phân trên mảng ứng viên đã sắp xếp tránh được lỗi biên.
- Cẩn thận ca suy biến: `source == target`, đồ thị không liên thông, `edges` rỗng.

## 15. Biến thể của bài

1. **k = 0**: bài trở thành *minimax path* — tìm đường có cạnh lớn nhất nhỏ nhất (giải được bằng Kruskal/DSU: thêm cạnh tăng dần tới khi `source` và `target` cùng thành phần).
2. **Đồ thị có hướng**: thuật toán giữ nguyên, chỉ thêm cạnh một chiều.
3. **Ngưỡng là số thực / trọng số thực**: ứng viên vẫn là các trọng số.
4. **Cho trước threshold, hỏi k nhỏ nhất**: chỉ cần một lần BFS 0-1.
5. **Nhiều truy vấn (source, target, k)**: có thể dùng tìm nhị phân song song (parallel binary search) hoặc tiền xử lý.
6. Các bài cùng họ: *Path With Minimum Effort* (LC 1631), *Swim in Rising Water* (LC 778), *Minimum Cost to Make at Least One Valid Path in a Grid* (LC 1368, BFS 0-1).

In [25]:
# Biến thể 1: k = 0 bằng DSU (Kruskal), đối chiếu với lời giải chính
def k0_dsu(n, edges, source, target):
    if source == target:
        return 0
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for u, v, w in sorted(edges, key=lambda x: x[2]):
        parent[find(u)] = find(v)
        if find(source) == find(target):
            return w
    return -1

rng = random.Random(15)
for _ in range(200):
    n = rng.randint(1, 8)
    e = [[rng.randrange(n), rng.randrange(n), rng.randint(1, 20)] for _ in range(rng.randint(0, 10))]
    s, t = rng.randrange(n), rng.randrange(n)
    assert k0_dsu(n, e, s, t) == Solution().minimumThreshold(n, e, s, t, 0)
print("Biến thể k=0 bằng DSU khớp lời giải chính.")

Biến thể k=0 bằng DSU khớp lời giải chính.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn đường đơn | Liệt kê mọi đường, lấy cạnh lớn thứ k+1 | hàm mũ | `O(n+m)` | Chỉ để kiểm thử, `n ≤ 8` |
| 2. Thử mọi ngưỡng | BFS 0-1 cho từng ứng viên | `O(m(n+m))` | `O(n+m)` | `m` nhỏ, muốn code đơn giản |
| 3. Nhị phân + BFS 0-1 | Đơn điệu + quyết định bằng 0-1 BFS | `O((n+m) log m)` | `O(n+m)` | **Lời giải chuẩn** |
| 4. Dijkstra phân tầng | Minimax với ≤ k lần miễn | `O(k m log(km))` | `O(nk)` | `k` nhỏ; khi cần biết thêm thông tin theo từng `j` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** Với `n = 3, edges = [[0,1,10],[1,2,1]], source = 0, target = 2`, đáp án khi `k = 0` và `k = 1` là gì?

**Bài 2 (tính tay).** Với ví dụ 1 và `k = 0`, tính tay đáp án (gợi ý: chỉ có một đường đi đơn).

**Bài 3 (điền TODO).** Hoàn thiện hàm `min_heavy_count(n, edges, source, target, T)` trả về số cạnh nặng ít nhất (hoặc `-1` nếu không tới được).

**Bài 4 (tự viết lại).** Viết lại lời giải chính nhưng tìm nhị phân trên **giá trị** `[0, max_w]` thay vì mảng ứng viên.

**Bài 5 (mini challenge).** Biến thể: thay vì "tối đa `k` cạnh nặng", đường hợp lệ nếu **tổng trọng số các cạnh nặng** `≤ B`. Viết hàm `min_threshold_budget(n, edges, source, target, B)` (gợi ý: với `T` cố định, chạy Dijkstra với chi phí `w` nếu nặng, `0` nếu nhẹ).

In [26]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def min_heavy_count(n, edges, source, target, T):
    # TODO: dựng danh sách kề, chạy BFS 0-1 với chi phí [w > T]
    # TODO: trả về dist[target] hoặc -1
    pass

In [27]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def min_threshold_budget(n, edges, source, target, B):
    # TODO: ứng viên = {0} ∪ {w}; với mỗi T, Dijkstra chi phí (w if w > T else 0)
    # TODO: tìm T nhỏ nhất sao cho dist[target] <= B
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chỉ có một đường `0-1-2`. (2) `k = 0`: mọi cạnh phải nhẹ. (3) `k = 1`: miễn cạnh lớn nhất, còn lại cạnh 1 → đáp án `1`.

**Bài 2.** (1) Đường duy nhất có trọng số `[5, 2, 4]`. (2) `k = 0` nghĩa là ngưỡng ≥ mọi cạnh. (3) Đáp án là max = `5`.

**Bài 3.** (1) Dùng `build_adj`. (2) Chi phí `1 if w > T else 0`. (3) Nhớ đổi `inf` thành `-1`.

**Bài 4.** (1) Kiểm tra liên thông bằng `T = max_w`. (2) `lo, hi = 0, max_w`. (3) `while lo < hi`, `hi = mid` khi `ok`.

**Bài 5.** (1) Tính đơn điệu vẫn đúng: tăng `T` chỉ làm chi phí giảm. (2) Chi phí không còn 0/1 → dùng Dijkstra. (3) Tìm nhị phân trên ứng viên như Cách 3.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1 & 2
ans1 = (Solution().minimumThreshold(3, [[0,1,10],[1,2,1]], 0, 2, 0),
        Solution().minimumThreshold(3, [[0,1,10],[1,2,1]], 0, 2, 1))
assert ans1 == (10, 1)
ans2 = Solution().minimumThreshold(*examples[0][:4], 0)
assert ans2 == 5
print("Bài 1:", ans1, "| Bài 2:", ans2)

Bài 1: (10, 1) | Bài 2: 5


In [29]:
# Bài 3
def min_heavy_count_sol(n, edges, source, target, T):
    d = heavy_dist(n, build_adj(n, edges), source, target, T)
    return -1 if d == float('inf') else d

assert min_heavy_count_sol(*examples[0][:4], 4) == 1
assert min_heavy_count_sol(*examples[0][:4], 2) == 2
assert min_heavy_count_sol(*examples[1][:4], 100) == -1
print("Bài 3 OK")

Bài 3 OK


In [30]:
# Bài 4
def sol_value_bs(n, edges, source, target, k):
    if source == target:
        return 0
    adj = build_adj(n, edges)
    if not edges:
        return -1
    hi = max(w for _, _, w in edges)
    if heavy_dist(n, adj, source, target, hi) == float('inf'):
        return -1
    lo = 0
    while lo < hi:
        mid = (lo + hi) // 2
        if heavy_dist(n, adj, source, target, mid) <= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

rng = random.Random(19)
for _ in range(300):
    n = rng.randint(1, 7)
    e = [[rng.randrange(n), rng.randrange(n), rng.randint(1, 15)] for _ in range(rng.randint(0, 9))]
    s, t, k = rng.randrange(n), rng.randrange(n), rng.randint(0, 4)
    assert sol_value_bs(n, e, s, t, k) == Solution().minimumThreshold(n, e, s, t, k)
print("Bài 4 OK")

Bài 4 OK


In [31]:
# Bài 5
def min_threshold_budget_sol(n, edges, source, target, B):
    if source == target:
        return 0
    adj = build_adj(n, edges)
    def cost(T):
        INF = float('inf')
        dist = [INF] * n; dist[source] = 0
        h = [(0, source)]
        while h:
            d, u = heapq.heappop(h)
            if d > dist[u]:
                continue
            for v, w in adj[u]:
                nd = d + (w if w > T else 0)
                if nd < dist[v]:
                    dist[v] = nd
                    heapq.heappush(h, (nd, v))
        return dist[target]
    cand = sorted({0} | {w for _, _, w in edges})
    if cost(cand[-1]) == float('inf'):
        return -1
    lo, hi = 0, len(cand) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if cost(cand[mid]) <= B:
            hi = mid
        else:
            lo = mid + 1
    return cand[lo]

# kiểm chứng bằng vét cạn đường đơn
def brute_budget(n, edges, s, t, B):
    adj = build_adj(n, edges); best = [None]; vis = [False] * n
    def dfs(u, ws):
        if u == t:
            for T in sorted({0} | set(ws)):
                if sum(w for w in ws if w > T) <= B:
                    if best[0] is None or T < best[0]:
                        best[0] = T
                    break
            return
        for v, w in adj[u]:
            if not vis[v]:
                vis[v] = True; ws.append(w); dfs(v, ws); ws.pop(); vis[v] = False
    vis[s] = True; dfs(s, [])
    return -1 if best[0] is None else best[0]

rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 6)
    e = [[rng.randrange(n), rng.randrange(n), rng.randint(1, 10)] for _ in range(rng.randint(0, 8))]
    s, t, B = rng.randrange(n), rng.randrange(n), rng.randint(0, 15)
    assert min_threshold_budget_sol(n, e, s, t, B) == brute_budget(n, e, s, t, B)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Template: nhị phân trên ứng viên + BFS 0-1
cand = sorted({0} | {w for _, _, w in edges})
def ok(T):                      # số cạnh nặng ít nhất <= k ?
    dist = [INF]*n; dist[s] = 0; dq = deque([s])
    while dq:
        u = dq.popleft()
        for v, w in adj[u]:
            c = int(w > T)
            if dist[u] + c < dist[v]:
                dist[v] = dist[u] + c
                dq.append(v) if c else dq.appendleft(v)
    return dist[t] <= k
if s == t: return 0
if not ok(cand[-1]): return -1
lo, hi = 0, len(cand)-1
while lo < hi:
    mid = (lo+hi)//2
    if ok(cand[mid]): hi = mid
    else: lo = mid + 1
return cand[lo]
```

**Checklist:**

- [ ] Đã xử lý `source == target` → `0`?
- [ ] Đã thêm `0` vào ứng viên và xử lý `edges` rỗng?
- [ ] Đồ thị vô hướng: thêm cạnh **hai chiều**?
- [ ] Điều kiện nặng là `w > T` (không phải `≥`)?
- [ ] Dùng BFS 0-1 (cho phép cập nhật lại `dist`), không phải BFS thường?
- [ ] Kiểm tra không liên thông → `-1` trước khi tìm nhị phân?
- [ ] Vòng nhị phân dạng `while lo < hi` với `hi = mid` / `lo = mid + 1`?